# 04 — Theme Analysis

Word frequency analysis across incident and RI text fields.

**Learning notes:**
- Word frequency is a useful *starting point* before investing in full NLP. It surfaces obvious patterns cheaply.
- A word appearing frequently in one team's incidents but not others is a signal of a team-specific systemic problem.
- A recurring word that appears in incidents but is NOT mentioned in any control description may indicate a control gap.
- This is NOT sentiment analysis or topic modelling — just counting words.

**Key questions:**
- What words recur most across incident titles and descriptions?
- Are those themes concentrated in specific teams?
- Do the recurring themes appear in control descriptions (suggesting coverage) or not (suggesting a gap)?

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.helpers import load_data, set_plot_style, save_processed, extract_themes

set_plot_style()

data = load_data("../data/raw")
incidents = data["incidents"]
ris       = data["ris"]
controls  = data["controls"]

print("Data loaded.")

## 1. Prepare Text

Combine title and description into a single text field for each incident and RI.

In [ ]:
incidents["combined_text"] = (
    incidents["title"].fillna("") + " " +
    incidents["description"].fillna("")
)
ri_text = ris["title"].fillna("") + " " + ris["description"].fillna("")

print(f"Incident text rows : {len(incidents['combined_text'])}")
print(f"RI text rows       : {len(ri_text)}")

## 2. Top Words in Incident Text

In [ ]:
incident_themes = extract_themes(incidents["combined_text"], top_n=25)
display(incident_themes.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(incident_themes["word"][::-1], incident_themes["count"][::-1])
ax.set_title("Top 25 Words — Incident Titles & Descriptions")
ax.set_xlabel("Frequency")
plt.tight_layout()
plt.savefig("../data/processed/chart_incident_themes.png", dpi=120)
plt.show()

## 3. Top Words in RI Text

In [ ]:
ri_themes = extract_themes(ri_text, top_n=25)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(ri_themes["word"][::-1], ri_themes["count"][::-1])
ax.set_title("Top 25 Words — RI Titles & Descriptions")
ax.set_xlabel("Frequency")
plt.tight_layout()
plt.savefig("../data/processed/chart_ri_themes.png", dpi=120)
plt.show()

## 4. Theme × Team Heatmap

For the top 10 incident words, count how many incidents per team contain that word.
High values in a single team cell = that team has a concentration of that theme.

In [ ]:
top_words = incident_themes["word"].head(10).tolist()

rows = []
for word in top_words:
    mask = incidents["combined_text"].str.contains(word, case=False, na=False)
    team_counts = incidents[mask]["team"].value_counts().to_dict()
    for team, count in team_counts.items():
        rows.append({"word": word, "team": team, "count": count})

theme_team_df = pd.DataFrame(rows)
pivot = theme_team_df.pivot_table(index="team", columns="word", values="count", fill_value=0)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, cmap="Blues", annot=True, fmt="d", ax=ax)
ax.set_title("Theme Frequency by Team  (incident text)")
plt.tight_layout()
plt.savefig("../data/processed/chart_theme_team_heatmap.png", dpi=120)
plt.show()

## 5. Themes vs Control Domains

Do the top incident keywords appear in control descriptions?
High overlap = the control framework addresses those themes.
Missing overlap = potential gap in the control framework.

In [ ]:
rows = []
for word in top_words:
    mask = controls["control_description"].str.contains(word, case=False, na=False)
    matched = controls[mask]["control_domain"].value_counts().to_dict()
    for domain, count in matched.items():
        rows.append({"word": word, "domain": domain, "count": count})

if rows:
    theme_ctrl_df = pd.DataFrame(rows)
    pivot2 = theme_ctrl_df.pivot_table(index="domain", columns="word", values="count", fill_value=0)

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(pivot2, cmap="Greens", annot=True, fmt="d", ax=ax)
    ax.set_title("Incident Themes vs Control Domains")
    plt.tight_layout()
    plt.savefig("../data/processed/chart_theme_control_heatmap.png", dpi=120)
    plt.show()
else:
    print("None of the top incident words appear in any control description.")
    print("This suggests a terminology gap between how incidents are described and how controls are written.")

## 6. Save Theme Frequencies

In [ ]:
path = save_processed(incident_themes, "theme_frequencies.csv", "../data/processed")
print(f"Saved: {path}")

## Key Findings

*(Fill in after running)*

- **Top 5 incident themes:** 
- **Teams with concentrated themes:** 
- **Themes NOT covered in any control domain:** 

---
**Next:** Run `05_summary_report.ipynb` to generate the final report and Excel export.